# 5.1 Survey Basics

**Survey series, class 1.** Most *All of Us* participants answer **surveys**: about their background, health, habits, family, and life circumstances. Survey answers tell us things health records rarely do (smoking, income, family history), but they have their own structure and their own traps. In this class we learn how the survey data is organised, how to find the questions we need, and how to turn raw answers into a clean table with **one row per person**.

### By the end of this notebook you'll be able to

1. Name the *All of Us* surveys, and explain why each was answered by a different number of people
2. Find questions by **text** and by **concept ID**, in the Workbench and in the public **Survey Explorer**
3. Tell the **non-answers** apart (*PMI: Skip*, *Prefer Not To Answer*, *Dont Know*, …), and know when *PMI: None* or *PMI: Other* is a **real** answer
4. Explain **skip logic**: why a missing answer often means "never asked"
5. Handle questions with **several answers** per person (select all that apply)
6. Build a tidy table with one row per person, and compare cases and controls

### 🔎 Analysis at a glance

| | |
|---|---|
| ❓ **Question** | Which surveys did the people in our type 2 diabetes cohort answer, and how do we turn their answers into usable variables? |
| 🧪 **The analysis** | Four BigQuery queries (the surveys, a question dictionary, who answered what, five example questions); counts and percents by group |
| 🎯 **Why** | Survey answers are easy to **misread**: a missing answer, a skipped answer, and "prefer not to answer" mean different things, and treating them all as "no" or "missing" biases every result |
| ✅ **What we conclude** | Which surveys and questions we can use, how complete they are in cases and controls, and how to code them |
| ⚠️ **What it can't tell us** | Anything about cause. And people who answer later, optional surveys aren't a random sample of participants |
| 🔗 **Connections** | Uses the cohort sample from **1.1**; builds on cleaning (**2.1**), reshaping (**2.2**), and Table 1 (**3.1**); **5.2** and **5.3** use these skills for family history, lifestyle, and social factors |

| | |
|---|---|
| 📂 **Uses** | the *All of Us* Controlled Tier dataset (BigQuery); `cohort_sample.csv` (from **1.1**; made for you if it's missing) |
| 💾 **Saves** | `survey_questions.csv` (a dictionary of every survey question), `survey_basics.csv` (one row per person) |

### Class plan

| Part | Minutes |
|---|---|
| 1. Key concepts: surveys, questions, answers, non-answers, skip logic | 15 |
| 2. Setup | 5 |
| 3. The ten surveys, and who answered them | 15 |
| 4. Finding questions | 15 |
| 5. Answers, non-answers, several answers, skip logic | 20 |
| 6. One row per person | 10 |
| Wrap-up | 10 |

### ⏱️ Time and 💰 cost

| | |
|---|---|
| ⏱️ **Class time** | 90 minutes |
| ⏱️ **Run time** | about **5 minutes** to run every cell. Slowest: the question dictionary (Part 4) |
| 💰 **Data cost** | about **$1 to $2**: four queries on `ds_survey`, a large table. BigQuery charges for the **columns** a query reads, even with a `WHERE` filter |
| 💰 **App cost** | about **$0.27 an hour** while the app is running. The app's disk costs about **$0.70 a day**, even while the app is stopped |
| 💡 **To spend less** | the question dictionary is saved to your bucket: search it in R (free) instead of querying again |
| **Estimated total** | about **$2** per student for this class |

> ⚠️ **Keep All of Us data in the Workbench.** Don't download, screenshot, or share participant-level data, or counts of 1 to 20. Clear outputs before you share or commit this notebook (**Edit → Clear Outputs of All Cells**).

## Part 1. Key concepts

### How the surveys work

Participants answer surveys online (or with staff), in **modules**: each survey is a separate module, taken at a different time.

- **The Basics**, **Overall Health**, and **Lifestyle** are answered when people **join**, so almost everyone has them.
- The others (family history, health care access, social factors, emotional health, …) are offered **later** and are optional, so **fewer** people answer them. Who answers later surveys isn't random: it depends on age, health, internet access, and how engaged people stay.

### Words you'll see

| Word | Means | Example |
|---|---|---|
| **Survey** (module) | one questionnaire | *Lifestyle* |
| **Question** | one item, with a **concept ID** and a **concept code** | 1585857, `Smoking_100CigsLifetime` |
| **Answer** | the option chosen, also with a concept ID | *Yes* |
| **Select all that apply** | a question where people can pick **several** answers: **one row per answer** | race: *White* **and** *Asian* |
| **Skip logic** (branching) | follow-up questions are shown only after certain answers | "How often do you smoke now?" is only asked of people who have smoked 100 cigarettes |

### Where the answers are stored

| Table | One row per… | Use it for |
|---|---|---|
| `ds_survey` | answer | **this series**: questions and answers as readable text, plus their concept IDs |
| `observation` | answer (among many other kinds of observation) | the standard OMOP version; the same answers, as concept IDs |

### Non-answers

Instead of an answer, a row can hold a **non-answer** code that starts with `PMI:` (*PMI* is the program's former name, the Precision Medicine Initiative):

| Code | Means | Usually treat as |
|---|---|---|
| *PMI: Skip* | saw the question, and skipped it | missing |
| *PMI: Prefer Not To Answer* | chose "prefer not to answer" | missing, or its own category (who prefers not to say can matter) |
| *PMI: Dont Know*, *PMI: Not Sure* | chose "don't know" or "not sure" | missing, or its own category |
| *PMI: None* | chose "none of these" | often a **real answer** (e.g. *none* of these conditions) |
| *PMI: Other* | chose "other" (any free text isn't released) | a **real answer** |
| *PMI: Does Not Apply To Me* | chose "does not apply" | a **real answer** |
| **no row at all** | **not asked** (skip logic), or never took the survey | depends on why: **check** |

### Where to read the questions

- **Survey Explorer** (public): every survey's questions, answer options, and branching, plus the **Survey Data Codebooks**: [researchallofus.org/data-tools/survey-explorer](https://www.researchallofus.org/data-tools/survey-explorer/)
- **Data Browser** (public): how many participants gave each answer: [databrowser.researchallofus.org](https://databrowser.researchallofus.org/)
- **In the Workbench**: the question dictionary we build in Part 4

## Part 2. Setup

`load_cohort_sample()` (in `aou_setup.R`) loads the 5,000 cases and 5,000 controls from **1.1**. If `cohort_sample.csv` isn't in your bucket yet, it runs the cohort query and makes the same sample (cents to under $1, 1–2 minutes).

In [ ]:
source("../aou_setup.R")  # settings, packages, and helper functions

cohort_sample <- load_cohort_sample()
cohort_sample %>% count(cohort)

ids_sql <- str_c(cohort_sample$person_id, collapse = ", ")   # our IDs, for WHERE person_id IN (...)

**🔍 Read the output:** 5,000 cases and 5,000 controls. `ids_sql` is the list of IDs for the queries below. **Don't print it**: it contains participant IDs.

### What's in `ds_survey`?

`INFORMATION_SCHEMA.COLUMNS` lists a table's columns. It reads no data, so it's free.

In [ ]:
run_query(str_glue("
  SELECT column_name, data_type
  FROM `{CURRENT_DATASET}`.INFORMATION_SCHEMA.COLUMNS
  WHERE table_name = 'ds_survey'
  ORDER BY ordinal_position"))

**🔍 Read the output:**

| Column | Means |
|---|---|
| `person_id` | the participant |
| `survey_datetime` | when they answered |
| `survey` | the survey's name |
| `question_concept_id`, `question` | the question, as an ID and as text |
| `answer_concept_id`, `answer` | the answer, as an ID and as text |
| `survey_version_concept_id`, `survey_version_name` | which **version** of the survey: some surveys were reworded over the years |

## Part 3. The ten surveys, and who answered them

### Across all of *All of Us*

How many people answered each survey, and how many questions each has? `COUNT(DISTINCT …)` counts different values.

💰 Under $0.50.

In [ ]:
survey_inventory <- run_query(str_glue("
  SELECT survey,
         COUNT(DISTINCT person_id)           AS n_people,
         COUNT(DISTINCT question_concept_id) AS n_questions
  FROM `{CURRENT_DATASET}.ds_survey`
  GROUP BY survey
  ORDER BY n_people DESC"))

survey_inventory

**🔍 Read the output:** ten surveys. The three answered at enrollment have the most people (in CDR v9: The Basics about 747,000, Overall Health and Lifestyle about 675,000); the later ones have far fewer (about 100,000 to 370,000).

- **Personal and Family Health History** has hundreds of "questions": each condition is asked as its own question ("who in your family has had type 2 diabetes?"). More in **5.2**.
- The **COVID-19** surveys were offered during the pandemic, to people who were participants then.

### In our sample

Which surveys did **our** 10,000 people answer, and when? One row per person per survey, with the first date they answered it. `DATE()` drops the time of day.

💰 Under $0.50.

In [ ]:
sample_surveys <- run_query(str_glue("
  SELECT person_id,
         survey,
         MIN(DATE(survey_datetime)) AS first_date
  FROM `{CURRENT_DATASET}.ds_survey`
  WHERE person_id IN ({ids_sql})
  GROUP BY person_id, survey")) %>%
  mutate(person_id = as.character(person_id)) %>%
  left_join(cohort_sample %>% select(person_id, cohort), by = "person_id")

# The percent of each group who answered each survey
survey_coverage <- sample_surveys %>%
  count(survey, cohort) %>%
  hide_small_counts() %>%                        # hide first, so no percent reveals a small count
  mutate(percent = round(100 * n / 5000, 1)) %>%
  pivot_wider(names_from = cohort, values_from = c(n, percent)) %>%
  arrange(desc(percent_case))

survey_coverage

**🔍 Read the output:** the percent of cases and of controls who answered each survey (out of 5,000 each).

- The three enrollment surveys: close to 100% in both groups.
- The later surveys: much lower, and **lower in cases**. In our test run, 57% of controls but 49% of cases answered Personal and Family Health History; for Social Determinants of Health, 50% and 42%. Any comparison that uses a later survey includes only part of each group, and a **different** part.

`hide_small_counts()` blanks out any count from 1 to 20, as *All of Us* policy requires for anything you share.

### When did people answer?

Answers are given on a **date**. That matters when we compare them with a diagnosis: an answer given **after** someone was diagnosed with diabetes can reflect the diagnosis (e.g. people who quit smoking after it). For each survey, the years in which our sample answered:

In [ ]:
sample_surveys %>%
  group_by(survey) %>%
  summarise(earliest_year = year(min(first_date)),
            median_year   = year(median(first_date)),
            latest_year   = year(max(first_date))) %>%
  arrange(median_year)

**🔍 Read the output:** the enrollment surveys span the whole program (2017 to 2024); later surveys started later (Social Determinants of Health in 2021, the two newest in 2023). The COVID-19 surveys are from 2020 to 2022. **5.2** compares answer dates with diagnosis dates.

### Who answers a later survey?

Compare people in our sample who did and didn't answer the **Social Determinants of Health** survey:

In [ ]:
sdoh_people <- sample_surveys %>%
  filter(survey == "Social Determinants of Health") %>%
  pull(person_id)

cohort_sample %>%
  mutate(answered_sdoh = person_id %in% sdoh_people) %>%
  group_by(answered_sdoh) %>%
  summarise(people         = n(),
            mean_age       = round(mean(age), 1),
            pct_female     = round(100 * mean(sex_at_birth == "Female", na.rm = TRUE), 1),
            pct_case       = round(100 * mean(cohort == "case"), 1))

**🔍 Read the output:** in our test run, the people who answered were a little **older** (mean age 60 vs 58), more often **female** (62% vs 57%), and less often **cases** (45% vs 54%). This is **selection**: any result that uses this survey describes the people who answered it, not the whole cohort.

**💬 Discuss:** why might someone **not** answer a later survey? (They joined after it was offered, they stopped using the app, they lack internet access, they were too ill, …) Which of these could be linked to diabetes?

## Part 4. Finding questions

There are over a thousand questions. We build a **question dictionary** once: every question, its survey, its concept code, and how many people answered it. Then we search it in R, for free.

The concept **code** comes from the `concept` table: a short name such as `Smoking_100CigsLifetime`, used in the codebooks.

💰 About $0.50: it reads the question text of every answer. It's saved to your bucket, so you only run it once.

In [ ]:
questions <- run_query(str_glue("
  WITH q AS (
    SELECT survey,
           question_concept_id,
           ANY_VALUE(question)       AS question,
           COUNT(DISTINCT person_id) AS n_people
    FROM `{CURRENT_DATASET}.ds_survey`
    GROUP BY survey, question_concept_id
  )
  SELECT q.*, c.concept_code
  FROM q
  LEFT JOIN `{CURRENT_DATASET}.concept` AS c ON c.concept_id = q.question_concept_id
  ORDER BY survey, n_people DESC"))

save_data(questions, "survey_questions.csv")
nrow(questions)

**🔍 Read the output:** 1,260 questions in our test run, saved as `survey_questions.csv`. `ANY_VALUE()` picks one question text per ID (the same ID can have slightly different wording in different survey versions).

`n_people` is a count across all of *All of Us*. A few questions were answered by very few people: don't share rows with `n_people` of 20 or fewer.

### Search by text

`find_questions()` keeps the questions whose text matches a word or pattern, ignoring upper and lower case:

In [ ]:
find_questions <- function(pattern) {
  questions %>%
    filter(str_detect(question, regex(pattern, ignore_case = TRUE))) %>%
    arrange(desc(n_people))
}

find_questions("smok")

**🔍 Read the output:** smoking questions from **Lifestyle** and the **COPE** survey.

- `n_people` shows the **skip logic**: everyone who took Lifestyle answered *100 Cigs Lifetime* (about 674,000), but only about 261,000 answered the follow-ups (*Smoke Frequency*, *Number Of Years*): the people who said *Yes*.
- The question text is short and coded (*Smoking: 100 Cigs Lifetime*); the Survey Explorer shows the full wording.
- Note each question's **concept ID**: queries should use the ID, not the text.

Now look for diabetes:

In [ ]:
find_questions("diabetes") %>% select(survey, question_concept_id, question, n_people)

**🔍 Read the output:** all from **Personal and Family Health History**, in two kinds:

- *Including yourself, who in your family has had type 2 diabetes? Select all that apply.*: answered by about 199,000 people, with answers like *Self*, *Mother*, *Father*. **5.2** uses it for family history.
- Follow-ups for people who said *Self* (age at diagnosis, treatment): about 36,000 people. Skip logic again.

### Search by concept ID

To check what an ID means (e.g. one you found in a paper or a codebook):

In [ ]:
questions %>% filter(question_concept_id %in% c(1585857, 1585711))

### ✍️ Your turn

Find the questions about **food** (e.g. worrying that food would run out). Which survey are they in, and how many people answered them?

<details>
<summary><b>Show answer</b></summary>

```r
find_questions("food") %>% select(survey, question_concept_id, question, n_people)
```

They're in **Social Determinants of Health**, a later survey, so far fewer people answered them than The Basics. **5.3** uses them.

</details>

In [ ]:
# Your code here

## Part 5. Answers, non-answers, several answers, skip logic

We download **five questions** for our sample, chosen because each shows a different feature:

| Name | Concept ID | Survey | Shows |
|---|---|---|---|
| `general_health` | 1585711 | Overall Health | one answer per person; an **ordered** scale |
| `race` | 1586140 | The Basics | **select all that apply**: several rows per person |
| `income` | 1585375 | The Basics | many **non-answers** (*Prefer Not To Answer*) |
| `smoked_100` | 1585857 | Lifestyle | the question that **opens** a follow-up |
| `smoke_frequency` | 1585860 | Lifestyle | the **follow-up**: only asked after *Yes* (skip logic) |

First check that the IDs are the questions we expect:

In [ ]:
QUESTIONS <- c(general_health  = 1585711,
               race            = 1586140,
               income          = 1585375,
               smoked_100      = 1585857,
               smoke_frequency = 1585860)

questions %>%
  filter(question_concept_id %in% QUESTIONS) %>%
  mutate(name = names(QUESTIONS)[match(question_concept_id, QUESTIONS)]) %>%
  select(name, survey, question_concept_id, concept_code, question)

**🔍 Read the output:** five rows, one per name, with matching question text and codes (e.g. `Smoking_SmokeFrequency`). Always run a check like this: a wrong ID gives a perfectly normal-looking answer to the **wrong question**.

Now the answers. 💰 About $0.50: BigQuery reads every column we ask for, for the whole table, before the `WHERE` filter.

In [ ]:
answers <- run_query(str_glue("
  SELECT person_id, question_concept_id, answer_concept_id, answer,
         DATE(survey_datetime) AS survey_date
  FROM `{CURRENT_DATASET}.ds_survey`
  WHERE question_concept_id IN ({str_c(QUESTIONS, collapse = ', ')})
    AND person_id IN ({ids_sql})")) %>%
  mutate(person_id = as.character(person_id),
         name      = names(QUESTIONS)[match(question_concept_id, QUESTIONS)])

answers %>% count(name)

# Answers repeat the question's short name: "General Health: Excellent". This keeps the part
# after the last ": " (e.g. "Excellent"), but leaves non-answers as they are ("PMI: Skip").
answer_text <- function(answer) {
  if_else(str_starts(answer, "PMI:"), answer, str_remove(answer, "^.*: "))
}

**🔍 Read the output:** the number of **rows** for each question. Compare them with 10,000: in our test run, `race` had **more** rows than people (10,846: several answers each), and `smoke_frequency` far **fewer** (4,107: only asked after *Yes*).

### 5a. Every answer, and the non-answers

`count(name, answer)` lists every answer to each question. `str_starts(answer, "PMI:")` flags the non-answers.

In [ ]:
answers %>%
  count(name, answer) %>%
  hide_small_counts() %>%
  print(n = 60)

**🔍 Read the output:**

- Answers repeat the question's short name before a colon (e.g. *General Health: Excellent*). `answer_text()`, defined above, removes that prefix but leaves non-answers as they are.
- **Income** has the most non-answers: in our test run, about 14% *Prefer Not To Answer* and 7% *Skip*. Income is a sensitive question.
- **Race** is one combined question about race **and** ethnicity: *Hispanic*, *MENA* (Middle Eastern or North African), and *NHPI* (Native Hawaiian or Other Pacific Islander) are answers alongside *White*, *Black*, *Asian*, and *AIAN* (American Indian or Alaska Native). *Race Ethnicity None Of These* is a **real** answer.
- **Smoking** has three kinds of non-answer: *Skip*, *Prefer Not To Answer*, and *Dont Know* (note: no apostrophe, so `"Don't Know"` would match nothing).
- Rows shown as `NA` in `n` had 1 to 20 people, hidden by `hide_small_counts()`.

How common are non-answers for each question, in each group?

In [ ]:
answers %>%
  left_join(cohort_sample %>% select(person_id, cohort), by = "person_id") %>%
  group_by(name, cohort) %>%
  summarise(rows          = n(),
            pct_nonanswer = round(100 * mean(str_starts(answer, "PMI:")), 1),
            .groups = "drop") %>%
  pivot_wider(names_from = cohort, values_from = c(rows, pct_nonanswer))

**🔍 Read the output:** `pct_nonanswer` is the percent of rows that are non-answers. For most questions it's 1–2%. **Income** is the exception, and it **differs by group**: in our test run, 24% of cases but 18% of controls gave no income. Dropping the non-answers would drop more cases than controls.

**💬 Discuss:** is someone who prefers not to give their income **like** the people who answered? (Often not: refusing can be more common at the top and the bottom of the income scale, so dropping them can bias results. One option is to keep *Prefer not to answer* as its own category.)

### 5b. One question, several answers: race

Race is **select all that apply**. How many rows does each person have?

In [ ]:
race_rows <- answers %>%
  filter(name == "race") %>%
  count(person_id, name = "answers_given")

race_rows %>% count(answers_given) %>% hide_small_counts()

**🔍 Read the output:** most people chose **one** answer, but some chose two or more (in our test run, about 7%), so they have several rows. If we joined `answers` to another table as it is, those people would be **duplicated** (the join problem from **2.3**).

Two ways to get **one row per person**:

1. **One column per answer** (`TRUE`/`FALSE`), with `pivot_wider()` from **2.2**. Keeps all the information.
2. **One summary value**: the answer, or *More than one* when there are several. Simpler, but it loses detail.

In [ ]:
# 1. One column per answer
race_wide <- answers %>%
  filter(name == "race") %>%
  mutate(answer = answer_text(answer), value = TRUE) %>%
  select(person_id, answer, value) %>%
  pivot_wider(names_from = answer, values_from = value, values_fill = FALSE)

ncol(race_wide)   # person_id + one column per answer

# 2. One summary value
race_one <- answers %>%
  filter(name == "race") %>%
  mutate(answer = answer_text(answer)) %>%
  group_by(person_id) %>%
  summarise(race_survey = if (n() > 1) "More than one" else first(answer))

race_one %>% count(race_survey) %>% hide_small_counts()

**🔍 Read the output:** `answer_text()` leaves just the answer (e.g. *White*). The first version has one column per answer; the second, one value per person, with *More than one* the fourth-largest group in our test run (after *White*, *Black*, and *Hispanic*). Many people who chose *Hispanic* also chose a race, so they move into *More than one*: whichever version you use, say how you handled several answers.

The cohort file already has a `race` column (from the `person` table). It's made from this same survey question, by the program, with its own rules for several answers. When you report race, say which version you used.

### 5c. Skip logic: smoking

*Have you smoked at least 100 cigarettes in your entire life?* Only people who answered **Yes** were then asked *How often do you now smoke cigarettes?*. Compare who has each answer:

In [ ]:
smoking <- answers %>%
  filter(name %in% c("smoked_100", "smoke_frequency")) %>%
  mutate(answer = answer_text(answer)) %>%
  select(person_id, name, answer) %>%
  pivot_wider(names_from = name, values_from = answer)

smoking %>%
  count(smoked_100, smoke_frequency) %>%
  hide_small_counts()

**🔍 Read the output:**

- *No* to 100 cigarettes → `smoke_frequency` is `NA`, every time. They were **never asked**: they're **never smokers**, not "missing".
- *Yes* → almost always a frequency answer (*Every Day*, *Some Days*, *Not At All*).
- *PMI: Skip* → many **did** get the follow-up, and answered it. Skipping a question that opens a follow-up still shows the follow-up.

So we **combine** the two questions into one variable, using the skip logic. `case_when()` checks each rule in order and uses the first that's true:

| Rule | Status |
|---|---|
| *No* to 100 cigarettes | **Never** |
| smokes *Every Day* or *Some Days* (whatever the first answer) | **Current** |
| *Yes* to 100 cigarettes, and *Not At All* now | **Former** |
| anything else | `NA` |

Why does *Former* need *Yes*, but *Current* doesn't? Someone who skipped the first question and smokes every day is clearly a current smoker. But someone who skipped it and answered *Not At All* could be a former smoker **or** someone who never smoked: we can't tell, so they're `NA`.

In [ ]:
smoking <- smoking %>%
  mutate(smoking_status = case_when(
    str_detect(smoked_100, regex("^no$", ignore_case = TRUE))                    ~ "Never",
    str_detect(smoke_frequency, regex("every day|some days", ignore_case = TRUE)) ~ "Current",
    str_detect(smoked_100, regex("^yes$", ignore_case = TRUE)) &
      str_detect(smoke_frequency, regex("not at all", ignore_case = TRUE))        ~ "Former",
    TRUE                                                                          ~ NA_character_))

smoking %>% count(smoking_status) %>% hide_small_counts()

**🔍 Read the output:** three smoking groups, plus `NA` for non-answers and the unclear combinations. In our test run: about 56% never, 26% former, and 14% current smokers; about 2% `NA`.

⚠️ If we had used `smoke_frequency` alone, every never-smoker would look **missing**, and `drop_na()` (from **2.1**) would have removed them: most of the sample, and not at random.

### 5d. One row per person per question?

Except for select-all questions, each person should have **one** answer to each question. Check:

In [ ]:
answers %>%
  filter(name != "race") %>%
  count(name, person_id) %>%
  filter(n > 1) %>%
  count(name, name = "people_with_more_than_one_row")

**🔍 Read the output:** ideally no rows (or very few). People who answered a survey **twice** (e.g. a newer version) would show up here. If so, keep the **latest** answer, using `survey_date`:

```r
answers %>% group_by(person_id, name) %>% slice_max(survey_date, n = 1, with_ties = FALSE)
```

## Part 6. One row per person

We put the pieces together: one row per person, with each variable coded, non-answers set to `NA`, and the categories in a sensible **order** (`factor(levels = …)`).

In [ ]:
general_health <- answers %>%
  filter(name == "general_health", !str_starts(answer, "PMI:")) %>%
  transmute(person_id,
            general_health = factor(answer_text(answer),
                                    levels = c("Excellent", "Very Good", "Good", "Fair", "Poor")))

survey_basics <- cohort_sample %>%
  select(person_id, cohort, age, sex_at_birth) %>%
  left_join(general_health,                                  by = "person_id") %>%
  left_join(race_one,                                        by = "person_id") %>%
  left_join(smoking %>% select(person_id, smoking_status),   by = "person_id") %>%
  mutate(smoking_status = factor(smoking_status, levels = c("Never", "Former", "Current")))

nrow(survey_basics)   # should still be 10,000: one row per person
save_data(survey_basics, "survey_basics.csv")

survey_basics %>% count(general_health)

**🔍 Read the output:** 10,000 rows, so no one was duplicated by the joins. If `general_health` shows `NA` for an answer you expected (e.g. *Very Good*), the answer text is written differently: compare with the answers listed in 5a and fix the `levels`.

### Compare cases and controls

Self-rated general health is one of the strongest predictors of future illness in health research. How does it differ between cases and controls?

In [ ]:
health_by_group <- survey_basics %>%
  filter(!is.na(general_health)) %>%
  count(cohort, general_health) %>%
  hide_small_counts() %>%
  group_by(cohort) %>%
  mutate(percent = round(100 * n / sum(n, na.rm = TRUE), 1)) %>%
  ungroup()

health_by_group

ggplot(health_by_group, aes(x = cohort, y = percent, fill = general_health)) +
  geom_col() +
  scale_fill_brewer(palette = "RdYlBu", direction = -1) +
  labs(x = NULL, y = "Percent of people who answered", fill = "General health",
       title = "Self-rated general health, by group") +
  theme_minimal(base_size = 14)

**🔍 Read the output:** cases rate their health as *Fair* or *Poor* more often than controls (in our test run, 41% vs 20%), and *Excellent* far less often (3% vs 13%). That fits what we know about diabetes, but remember: cases are also **older** (**3.1**), and age affects self-rated health too. Comparing like with like needs a model (**3.2**), which **5.2** does.

### ✍️ Your turn

Make an **income** variable for `survey_basics`: remove the prefix, and keep *Prefer Not To Answer* as its **own category** instead of `NA`. Then compare cases and controls: who prefers not to give their income?

<details>
<summary><b>Show answer</b></summary>

```r
income <- answers %>%
  filter(name == "income") %>%
  transmute(person_id,
            income = case_when(
              answer == "PMI: Prefer Not To Answer" ~ "Prefer not to answer",
              str_starts(answer, "PMI:")            ~ NA_character_,
              TRUE                                  ~ answer_text(answer)))

survey_basics %>%
  left_join(income, by = "person_id") %>%
  count(cohort, income) %>%
  hide_small_counts() %>%
  group_by(cohort) %>%
  mutate(percent = round(100 * n / sum(n, na.rm = TRUE), 1)) %>%
  print(n = 40)
```

The income categories aren't in order yet (they sort as text): **5.3** puts them in order and uses them.

</details>

In [ ]:
# Your code here

## ⚠️ Common errors

| Message or problem | Means | Fix |
|---|---|---|
| A filter on `survey` or `question` returns **0 rows** | the text is written slightly differently | copy the text from the dictionary, or better, use concept IDs |
| A question has **more rows than people** | select all that apply, or answered twice | one column per answer, one summary value, or the latest answer (5b, 5d) |
| A follow-up question is mostly `NA` | skip logic: most people weren't asked | combine it with the question that opens it (5c) |
| A factor level is all `NA` | the answer text is written differently from your `levels` | compare with `count(answer)` and fix the spelling |
| *could not find function "run_query"* | `aou_setup.R` isn't loaded | run the first code cell |

## ✅ Key takeaways

- *All of Us* has **ten surveys**; later ones were answered by fewer, and **different**, people
- Find questions in a **dictionary** (or the Survey Explorer), then query by **concept ID**, not text
- **Non-answers** start with `PMI:`; some (*None*, *Other*, *Does Not Apply To Me*) are real answers
- A **missing** answer can mean "never asked": use the **skip logic**
- **Select-all** questions give several rows per person: reshape before joining
- Answers have **dates**: check whether they came before or after a diagnosis

**Next:** **5.2 family_history_lifestyle**: family history of diabetes, smoking, and alcohol, in cases and controls.

**Before you share or commit this notebook:** clear all outputs (**Edit → Clear Outputs of All Cells**).

## 🧑‍🏫 Instructor notes

- **Timing:** Part 1 is the heart of the class. If time is short, skip the "Who answers a later survey?" cell and the income exercise.
- **Who answers later surveys** is the best moment of the class: ask students to guess which group answers more before showing the table.
- **PMI** comes up every time: it's the program's former name, the Precision Medicine Initiative.
- **Race:** the `race` column in the cohort file comes from the same survey question, mapped by the program. Students often ask why the two differ: different rules for several answers, and non-answers.
- **Likely question:** "Can I use the `observation` table instead?" Yes: same answers, as concept IDs. `ds_survey` is easier to read.